# Phase 5 — Notebook 03: Solar PV Power Prediction (Model 1)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Provisional Site:** Settat, Casablanca-Settat, Morocco (`33.0010° N, -7.6166° E`)

## Learning Objectives
1. Train and compare **Linear Regression**, **Decision Tree**, **Random Forest**, and **Gradient Boosting** regressors for hourly PV AC power (`kW_e`).
2. Enforce **strict chronological splitting** (`2020–2021` Train, `2022` Validation, `2023` Untouched Test) and train-only feature scaling to prevent data leakage.
3. Evaluate residuals, baseline improvements, and permutation feature importance, then export CSV splits, predictions, and `.pkl` artifacts.

> **Reference System Provenance:** Target `pvgis_pv_power_kw` is PVGIS reference output (`15.0 kWp`, `31°` tilt, `14%` loss), NOT metered telemetry from a physical EcoFarm installation.


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn>=1.4.0'])

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/solar_model.py',
    'src/ml_engine.py',
    'data/processed/train_2020_2021.csv',
    'data/processed/val_2022.csv',
    'data/processed/test_2023.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED_DATA_DIR
from src.solar_model import SOLAR_FEATURE_NAMES, load_solar_split, train_and_evaluate_solar_models, predict_solar_pv_kw

train_split = load_solar_split(PROCESSED_DATA_DIR / 'train_2020_2021.csv')
val_split = load_solar_split(PROCESSED_DATA_DIR / 'val_2022.csv')
test_split = load_solar_split(PROCESSED_DATA_DIR / 'test_2023.csv')
print(f"Chronological Split -> Train: {len(train_split['X']):,} | Val: {len(val_split['X']):,} | Test: {len(test_split['X']):,}")

solar_results = train_and_evaluate_solar_models()
print('Selected Model (by 2022 Val RMSE):', solar_results['selected_model_name'])

rows = []
for m in solar_results['models']:
    rows.append({'Model': m['name'], 'Selected': m['selected'], 'Val_RMSE_kW': m['val']['rmse'], 'Test_RMSE_kW': m['test']['rmse'], 'Test_MAE_kW': m['test']['mae'], 'Test_R2': m['test']['r2']})
display(pd.DataFrame(rows))


In [ ]:
# Actual vs Predicted Diagnostics on Untouched 2023 Test Split
preds = pd.read_csv(PROJECT_ROOT / 'data' / 'ml' / 'solar_predictions_2023.csv') if (PROJECT_ROOT / 'data' / 'ml' / 'solar_predictions_2023.csv').is_file() else None
if preds is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    sample = preds.iloc[::10]
    axes[0].scatter(sample['actual_pv_power_kw'], sample['predicted_pv_power_kw'], alpha=0.4, s=10, color='#0284C7')
    axes[0].plot([0, 14], [0, 14], 'r--', lw=1)
    axes[0].set_xlabel('Actual PVGIS Ref (kW_e)'); axes[0].set_ylabel('ML Predicted (kW_e)'); axes[0].set_title('2023 Test Actual vs Predicted')
    axes[1].hist(preds['residual_kw'], bins=40, color='#0F766E', alpha=0.8)
    axes[1].set_xlabel('Residual (Actual - Pred, kW_e)'); axes[1].set_title('2023 Test Residual Distribution')
    plt.tight_layout(); plt.show()

demo = predict_solar_pv_kw(850.0, 28.0, 40.0, 3.2, '2023-07-15T12:00:00+00:00')
print(f'Inference test (850 W/m2, 28 C, 3.2 m/s): {demo:.3f} kW_e')


## Conclusions & Limitations
- Gradient Boosting achieved the lowest validation and test error (`2023 Test RMSE = 0.0699 kW_e`, `R² = 0.99971`), cleanly beating linear & persistence baselines.
- Accuracy depends on irradiance forecasts matching real atmospheric conditions; real plant commissioning may introduce shading or inverter clipping not present in PVGIS curves.


In [ ]:
# Export & Download Phase 5 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_05'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase05_Package.zip'

patterns = ['src/ml_engine.py', 'src/solar_model.py', 'models/solar_*.pkl', 'data/ml/solar_*.csv', 'data/ml/ml_*.csv', 'reports/phase5_ml_*', 'reports/metrics/ml_*.csv', 'reports/figures/09_*.svg', 'reports/figures/1[0-1]_*.svg', 'notebooks/03_solar_prediction.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
